# Aula 3 — Testes Automatizados para Modelos de IA
## Testes de integração em pipelines + Validação de dados e verificação de comportamento dos modelos

IEC PUC Minas — Engenharia de Inteligência Artificial e MLOps

## Setup

In [1]:
!pip install -q "ipytest==0.14.*" "pandas>=2"

import ipytest
import pytest
import pandas as pd
ipytest.autoconfig()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 1.0 MB/s eta 0:00:00


---
## 1. Testes de integração em pipelines de ML

### Caso real: Knight Capital Group (2012)

In [6]:
def load_data():
    return pd.DataFrame({
        "user_id": [1, 2, 3],
        "score": [0.8, 0.3, 0.95],
        "category": ["A", "B", "A"],
    })


def clean_data(df):
    df = df.dropna()
    df = df.drop_duplicates()
    return df


def compute_features(df):
    df = df.copy()
    df["score_normalized"] = df["score"].clip(0, 1)
    return df

compute_features(clean_data(load_data()))

,user_id,score,category,score_normalized
0,1,0.80,A,0.80
1,2,0.30,B,0.30
2,3,0.95,A,0.95


### Teste de contrato entre estágios

`compute_features` espera a coluna `score`. Esse teste garante que `clean_data` nunca entrega um DataFrame sem ela — mesmo que alguém mude `clean_data` no futuro.

In [3]:
%%ipytest

def load_data():
    return pd.DataFrame({
        "user_id": [1, 2, 3],
        "score": [0.8, 0.3, 0.95],
        "category": ["A", "B", "A"],
    })


def clean_data(df):
    df = df.dropna()
    df = df.drop_duplicates()
    return df


def test_clean_data_preserves_required_columns():
    df = load_data()
    cleaned = clean_data(df)
    assert {"user_id", "score", "category"} <= set(cleaned.columns)


.                                                                                            [100%]
1 passed in 0.01s


### O contrato "óbvio" não é suficiente

Um teste de contrato só pega o que ele verifica. Se `clean_data` fosse trocada por uma versão desatualizada que "esqueceu" de remover duplicatas — o tipo de inconsistência que aconteceria se só parte dos servidores recebesse uma atualização, como no caso Knight Capital — o teste de contrato acima **ainda passa**, porque só checa as colunas.

In [8]:
%%ipytest

def load_data_com_duplicata():
    return pd.DataFrame({
        "user_id": [1, 2, 2, 3],
        "score": [0.8, 0.3, 0.3, 0.95],
        "category": ["A", "B", "B", "A"],
    })


def clean_data_desatualizada(df):
    # versão "presa no passado" -- esqueceu de remover duplicatas
    df = df.dropna()
    return df


def test_contrato_de_colunas_nao_detecta_duplicata():
    df = load_data_com_duplicata()
    cleaned = clean_data_desatualizada(df)
    # passa: as colunas certas continuam lá
    assert {"user_id", "score", "category"} <= set(cleaned.columns)
    # mas um teste de integração completo pegaria isso:
    assert not cleaned.duplicated(subset="user_id").any(), (
        "Duplicata sobrevivente -- exatamente o tipo de inconsistência que um "
        "componente 'desatualizado' no pipeline provocaria (caso Knight Capital: "
        "servidor sem a atualização reagindo diferente dos outros 7)"
    )

F                                                                                            [100%]
============================================= FAILURES =============================================
__________________________ test_contrato_de_colunas_nao_detecta_duplicata __________________________

    def test_contrato_de_colunas_nao_detecta_duplicata():
        df = load_data_com_duplicata()
        cleaned = clean_data_desatualizada(df)
        # passa: as colunas certas continuam lá
        assert {"user_id", "score", "category"} <= set(cleaned.columns)
        # mas um teste de integração completo pegaria isso:
>       assert not cleaned.duplicated(subset="user_id").any(), (
            "Duplicata sobrevivente -- exatamente o tipo de inconsistência que um "
            "componente 'desatualizado' no pipeline provocaria (caso Knight Capital: "
            "servidor sem a atualização reagindo diferente dos outros 7)"
        )
E       AssertionError: Duplicata sobrevivente -- exa

### Teste de integração do pipeline completo

In [5]:
%%ipytest

def load_data():
    return pd.DataFrame({
        "user_id": [1, 2, 3],
        "score": [0.8, 0.3, 0.95],
        "category": ["A", "B", "A"],
    })


def clean_data(df):
    df = df.dropna()
    df = df.drop_duplicates()
    return df


def compute_features(df):
    df = df.copy()
    df["score_normalized"] = df["score"].clip(0, 1)
    return df


def test_pipeline_end_to_end():
    df = load_data()
    df = clean_data(df)
    df = compute_features(df)
    assert "score_normalized" in df.columns
    assert df["score_normalized"].between(0, 1).all()


.                                                                                            [100%]
1 passed in 0.01s


---
## 2. Validação de dados


Aqui usamos `pytest.raises`. Vamos verificar que a validação **rejeita corretamente** um dado ruim.

**Atenção:** o argumento `match=` do `pytest.raises` é uma expressão regular (regex), não um texto literal.

In [ ]:
%%ipytest

EXPECTED_COLUMNS = {"user_id": "int64", "score": "float64", "category": "object"}


def validate_schema(df, expected_columns=EXPECTED_COLUMNS):
    missing = set(expected_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Colunas faltando: {missing}")
    for col, dtype in expected_columns.items():
        if str(df[col].dtype) != dtype:
            raise ValueError(f"Coluna {col} tem tipo {df[col].dtype}, esperado {dtype}")
    return True


def test_validate_schema_accepts_valid_dataframe():
    df = pd.DataFrame({"user_id": [1], "score": [0.5], "category": ["A"]})
    assert validate_schema(df) is True


def test_validate_schema_rejects_missing_column():
    df = pd.DataFrame({"user_id": [1], "score": [0.5]})  # falta 'category'
    with pytest.raises(ValueError, match="Colunas faltando"):
        validate_schema(df)


.

.

                                                                                           [100%]


2 passed in 0.01s


In [ ]:
%%ipytest

def check_no_missing_values(df, columns):
    for col in columns:
        if df[col].isna().any():
            raise ValueError(f"Coluna {col} tem valores ausentes")
    return True


def check_score_range(df, column="score", lo=0.0, hi=1.0):
    # dropna() primeiro: valores ausentes são problema de completude,
    # não de faixa -- quem checa isso é o check_no_missing_values
    if not df[column].dropna().between(lo, hi).all():
        raise ValueError(f"Coluna {column} tem valores fora do intervalo [{lo}, {hi}]")
    return True


def check_unique_ids(df, id_column="user_id"):
    if df[id_column].duplicated().any():
        raise ValueError(f"IDs duplicados em {id_column}")
    return True


def test_check_no_missing_values_rejects_nan():
    df = pd.DataFrame({"score": [0.5, None]})
    with pytest.raises(ValueError, match="valores ausentes"):
        check_no_missing_values(df, ["score"])


def test_check_score_range_rejects_out_of_range():
    df = pd.DataFrame({"score": [0.5, 1.5]})
    with pytest.raises(ValueError, match="fora do intervalo"):
        check_score_range(df)


def test_check_unique_ids_rejects_duplicates():
    df = pd.DataFrame({"user_id": [1, 2, 2]})
    with pytest.raises(ValueError, match="duplicados"):
        check_unique_ids(df)


.

.

.

                                                                                          [100%]


3 passed in 0.02s


### Quando `pytest.raises` não captura nada

In [ ]:
%%ipytest

def check_score_range(df, column="score", lo=0.0, hi=1.0):
    if not df[column].dropna().between(lo, hi).all():
        raise ValueError(f"Coluna {column} tem valores fora do intervalo [{lo}, {hi}]")
    return True


def test_demo_did_not_raise():
    df = pd.DataFrame({"score": [0.5, 0.8]})  # dados válidos -- nada deveria falhar
    with pytest.raises(ValueError):
        check_score_range(df)  # mas aqui exigimos que levante -- vai falhar com "DID NOT RAISE"


F

                                                                                            [100%]


============================================ FAILURES =============================================
_____________________________________ test_demo_did_not_raise _____________________________________

    def test_demo_did_not_raise():
        df = pd.DataFrame({"score": [0.5, 0.8]})  # dados válidos -- nada deveria falhar
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE <class 'ValueError'>

C:\Users\FHPA\AppData\Local\Temp\ipykernel_91272\1207117407.py:9: Failed
===================================== short test summary info =====================================
FAILED t_f8ba3dd44ba04efe92a28ac5e56b45fa.py::test_demo_did_not_raise - Failed: DID NOT RAISE <class 'ValueError'>
1 failed in 0.02s


---
## 3. Verificação de comportamento do modelo

In [ ]:
%%ipytest

def mock_predict(df):
    """Modelo mock: 'prediz' reaproveitando o score de entrada, já limitado a [0, 1]."""
    return df["score"].clip(0, 1)


# usa assert (não raise ValueError): aqui é uma invariante que verificamos
# dentro do próprio teste, não uma função de validação reutilizável chamada
# por outro código -- por isso o contrato de erro é diferente dos checks acima
def check_predictions_valid(predictions):
    assert predictions.notna().all(), "Predições não podem ter NaN"
    assert predictions.between(0, 1).all(), "Predições devem estar entre 0 e 1"
    return True


def test_predictions_are_valid():
    df = pd.DataFrame({"score": [0.2, 0.9, 0.5]})
    preds = mock_predict(df)
    assert check_predictions_valid(preds) is True


.

                                                                                            [100%]


1 passed in 0.01s


### Caso real: Google Flu Trends (2013)

### Sobre Great Expectations e Deepchecks

```python
# Great Expectations (ilustrativo)
validator.expect_column_values_to_be_between("score", min_value=0, max_value=1)
validator.expect_column_values_to_not_be_null("score")
```

A lógica é a mesma que já praticamos: descrever o que você espera dos dados, e deixar a ferramenta verificar e reportar.

---
## 4. Exercício

O dataset abaixo tem pelo menos 3 problemas plantados. Use `pytest.raises` para escrever um teste que comprove que cada validador encontra o problema certo.

In [ ]:
dirty_df = pd.DataFrame({
    "user_id": [1, 2, 2, 4],
    "score": [0.8, 1.5, 0.3, None],
    "category": ["A", "B", "B", "A"],
})
dirty_df

,user_id,score,category
0,1,0.8,A
1,2,1.5,B
2,2,0.3,B
3,4,NaN,A


In [ ]:
%%ipytest

dirty_df = pd.DataFrame({
    "user_id": [1, 2, 2, 4],
    "score": [0.8, 1.5, 0.3, None],
    "category": ["A", "B", "B", "A"],
})


def check_unique_ids(df, id_column="user_id"):
    if df[id_column].duplicated().any():
        raise ValueError(f"IDs duplicados em {id_column}")
    return True


def check_score_range(df, column="score", lo=0.0, hi=1.0):
    if not df[column].dropna().between(lo, hi).all():
        raise ValueError(f"Coluna {column} tem valores fora do intervalo [{lo}, {hi}]")
    return True


def check_no_missing_values(df, columns):
    for col in columns:
        if df[col].isna().any():
            raise ValueError(f"Coluna {col} tem valores ausentes")
    return True


def test_dirty_df_has_duplicate_ids():
    with pytest.raises(ValueError, match="duplicados"):
        check_unique_ids(dirty_df)


def test_dirty_df_has_score_out_of_range():
    with pytest.raises(ValueError, match="fora do intervalo"):
        check_score_range(dirty_df)


def test_dirty_df_has_missing_score():
    with pytest.raises(ValueError, match="valores ausentes"):
        check_no_missing_values(dirty_df, ["score"])


F

F

F

                                                                                          [100%]


============================================ FAILURES =============================================
_________________________________ test_dirty_df_has_duplicate_ids _________________________________

    def test_dirty_df_has_duplicate_ids():
>       pytest.fail("TODO: escreva o teste com pytest.raises")
E       Failed: TODO: escreva o teste com pytest.raises

C:\Users\FHPA\AppData\Local\Temp\ipykernel_91272\3860779505.py:29: Failed
______________________________ test_dirty_df_has_score_out_of_range _______________________________

    def test_dirty_df_has_score_out_of_range():
>       pytest.fail("TODO: escreva o teste com pytest.raises")
E       Failed: TODO: escreva o teste com pytest.raises

C:\Users\FHPA\AppData\Local\Temp\ipykernel_91272\3860779505.py:34: Failed
_________________________________ test_dirty_df_has_missing_score _________________________________

    def test_dirty_df_has_missing_score():
>       pytest.fail("TODO: escreva o teste com pytest.raises")
E       Fai

### Quando a limpeza não é suficiente

In [ ]:
def clean_data(df):
    df = df.dropna()
    df = df.drop_duplicates()
    return df


resultado = clean_data(dirty_df)
print(resultado)
print()
print("dropna() removeu a linha com score ausente.")
print("drop_duplicates() NÃO removeu o user_id=2 duplicado")
print("(as duas linhas têm score diferente, não são duplicatas EXATAS).")
print("E o score=1.5 (fora do intervalo) continua intacto.")
print()
print("Lição: 'limpar' dados não é o mesmo que 'validar' dados.")
print("clean_data() resolve alguns problemas estruturais, mas não")
print("garante que os dados fazem sentido -- é pra isso que existem")
print("os validadores explícitos.")

   user_id  score category
0        1    0.8        A
1        2    1.5        B
2        2    0.3        B

dropna() removeu a linha com score ausente.
drop_duplicates() NÃO removeu o user_id=2 duplicado
(as duas linhas têm score diferente, não são duplicatas EXATAS).
E o score=1.5 (fora do intervalo) continua intacto.

Lição: 'limpar' dados não é o mesmo que 'validar' dados.
clean_data() resolve alguns problemas estruturais, mas não
garante que os dados fazem sentido -- é pra isso que existem
os validadores explícitos.


---
## 5. Suíte completa

In [ ]:
%%ipytest -v

# --- código sob teste -------------------------------------------------

def load_data():
    return pd.DataFrame({
        "user_id": [1, 2, 3],
        "score": [0.8, 0.3, 0.95],
        "category": ["A", "B", "A"],
    })


def clean_data(df):
    df = df.dropna()
    df = df.drop_duplicates()
    return df


def compute_features(df):
    df = df.copy()
    df["score_normalized"] = df["score"].clip(0, 1)
    return df


EXPECTED_COLUMNS = {"user_id": "int64", "score": "float64", "category": "object"}


def validate_schema(df, expected_columns=EXPECTED_COLUMNS):
    missing = set(expected_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Colunas faltando: {missing}")
    for col, dtype in expected_columns.items():
        if str(df[col].dtype) != dtype:
            raise ValueError(f"Coluna {col} tem tipo {df[col].dtype}, esperado {dtype}")
    return True


def check_score_range(df, column="score", lo=0.0, hi=1.0):
    if not df[column].dropna().between(lo, hi).all():
        raise ValueError(f"Coluna {column} tem valores fora do intervalo [{lo}, {hi}]")
    return True


def mock_predict(df):
    return df["score"].clip(0, 1)


def check_predictions_valid(predictions):
    assert predictions.notna().all(), "Predições não podem ter NaN"
    assert predictions.between(0, 1).all(), "Predições devem estar entre 0 e 1"
    return True


# --- suíte de testes ----------------------------------------------------

def test_pipeline_end_to_end():
    df = compute_features(clean_data(load_data()))
    assert "score_normalized" in df.columns
    assert df["score_normalized"].between(0, 1).all()


def test_validate_schema_accepts_valid_dataframe():
    assert validate_schema(load_data()) is True


def test_validate_schema_rejects_missing_column():
    broken = load_data().drop(columns=["category"])
    with pytest.raises(ValueError, match="Colunas faltando"):
        validate_schema(broken)


def test_check_score_range_rejects_out_of_range():
    df = pd.DataFrame({"score": [0.5, 1.5]})
    with pytest.raises(ValueError, match="fora do intervalo"):
        check_score_range(df)


def test_predictions_are_valid():
    preds = mock_predict(load_data())
    assert check_predictions_valid(preds) is True


======================================= test session starts =======================================
platform win32 -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: C:\Users\FHPA\Documents\PUC Minas\Testes Automatizados IA\aula-3-20-08
plugins: anyio-4.11.0, Faker-37.11.0, hypothesis-6.164.0, langsmith-0.4.37, asyncio-1.3.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 5 items

t_f8ba3dd44ba04efe92a28ac5e56b45fa.py 

.

.

.

.

.

                                                  [100%]



======================================== 5 passed in 0.04s ========================================


---
## 6. [Prático] Onde a validação entraria — decidam e implementem

Completem o `run_pipeline` abaixo decidindo em que ponto(s) chamar os validadores. O teste no fim da célula é o critério: ele exige que o `dirty_df` seja **rejeitado** na hora de servir. Enquanto o `TODO` estiver vazio, ele falha.

In [ ]:
%%ipytest

# --- setup auto-contido: esta celula roda mesmo se voce pular as anteriores ---

def clean_data(df):
    df = df.dropna()
    df = df.drop_duplicates()
    return df


def compute_features(df):
    df = df.copy()
    df["score_normalized"] = df["score"].clip(0, 1)
    return df


def validate_schema(df, expected_columns=("user_id", "score", "category")):
    missing = set(expected_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Colunas faltando: {missing}")
    return True


def check_score_range(df, column="score", lo=0.0, hi=1.0):
    if not df[column].dropna().between(lo, hi).all():
        raise ValueError(f"Coluna {column} tem valores fora do intervalo [{lo}, {hi}]")
    return True


def check_unique_ids(df, id_column="user_id"):
    if df[id_column].duplicated().any():
        raise ValueError(f"IDs duplicados em {id_column}")
    return True


dirty_df = pd.DataFrame({
    "user_id": [1, 2, 2, 4],
    "score": [0.8, 1.5, 0.3, None],
    "category": ["A", "B", "B", "A"],
})


# --- exercício ---------------------------------------------------------------

def run_pipeline(df, mode):
    """mode = 'train' ou 'serve'.

    Validamos nos dois modos, ANTES de limpar e de gerar features.

    Por quê nos dois?
    - train: dado sujo treina modelo sujo (duplicata e score 1.5 entram no aprendizado).
    - serve: o teste (e a produção) exigem rejeitar, não "consertar em silêncio".

    Por quê antes do clean/features?
    - validate_schema primeiro: sem as colunas certas, o resto nem deveria rodar.
    - check_unique_ids / check_score_range no bruto: clean_data() não pega
      user_id duplicado com scores diferentes, e compute_features().clip()
      esconderia o 1.5 em score_normalized. Validar depois da "limpeza"
      seria aprovar o que a aula mostrou que continua errado.
    """
    validate_schema(df)
    check_unique_ids(df)
    check_score_range(df)

    df = clean_data(df)
    df = compute_features(df)
    return df


def test_run_pipeline_rejeita_dado_sujo():
    with pytest.raises(ValueError):
        run_pipeline(dirty_df, mode="serve")


F

                                                                                            [100%]


============================================ FAILURES =============================================
_______________________________ test_run_pipeline_rejeita_dado_sujo _______________________________

    def test_run_pipeline_rejeita_dado_sujo():
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE <class 'ValueError'>

C:\Users\FHPA\AppData\Local\Temp\ipykernel_91272\3939933639.py:56: Failed
===================================== short test summary info =====================================
FAILED t_f8ba3dd44ba04efe92a28ac5e56b45fa.py::test_run_pipeline_rejeita_dado_sujo - Failed: DID NOT RAISE <class 'ValueError'>
1 failed in 0.03s


---
>
## 7. [Prática 3] Contrato de unidade - o erro da Mars Climate Orbiter

A sonda se perdeu porque um módulo entregava impulso em **libra-força·segundo** e o
outro lia como **newton·segundo**. As duas funções estavam certas por dentro; o que
faltou foi alguém checar a fronteira.

Aqui o pipeline tem o mesmo defeito: `coleta_telemetria` devolve o impulso em `lbf*s`,
mas `aplica_correcao` assume `N*s` (1 lbf·s = 4,448 N·s).

**Sua tarefa:** escreva o teste de contrato que pega isso *antes* de a correção ser aplicada.

In [ ]:
%%ipytest

FATOR_LBF_PARA_N = 4.448221615

def coleta_telemetria():
    # o fornecedor do sensor entrega em libra-forca*segundo
    return {"impulso": 100.0, "unidade": "lbf*s"}


def aplica_correcao(payload):
    # este modulo assume, sem verificar, que o valor chega em newton*segundo
    return payload["impulso"] * 1.05


# TODO 1: escreva um teste de contrato que FALHE enquanto as unidades nao baterem.
# Dica: o contrato aqui e o campo payload["unidade"] == "N*s".
def test_contrato_unidade_do_impulso():
    pytest.fail("TODO: verifique que coleta_telemetria() entrega a unidade que aplica_correcao espera")


# TODO 2: implemente a conversao que conserta o contrato.
def converte_para_newton_segundo(payload):
    pytest.fail("TODO: devolva um novo payload com impulso em N*s e unidade 'N*s'")


# TODO 3: depois de implementar, este teste precisa passar.
def test_conversao_produz_unidade_esperada():
    pytest.fail("TODO: converta e verifique unidade == 'N*s' e o valor convertido")


# TODO 4: propriedade — converter nao pode mudar a grandeza fisica.
# 100 lbf*s viram 444.82 N*s; converter de novo NAO pode multiplicar outra vez.
def test_conversao_e_idempotente():
    pytest.fail("TODO: converter duas vezes tem que dar o mesmo que converter uma vez")

---
## 8. [Prática 4] Completude - as 15.841 linhas que sumiram

No Public Health England, o formato `.xls` parou em 65.536 linhas e **não avisou**.
Ninguém comparou quantas linhas entraram com quantas saíram.

`consolida_resultados` abaixo tem exatamente esse defeito, em escala pequena para caber
na aula: ela trunca silenciosamente no limite.

**Sua tarefa:** escreva o teste de completude que teria gritado na hora.

In [ ]:
%%ipytest

LIMITE_DE_LINHAS = 10  # o ".xls" desta aula


def consolida_resultados(blocos, limite=LIMITE_DE_LINHAS):
    juntos = []
    for bloco in blocos:
        juntos.extend(bloco)
    return juntos[:limite]  # trunca em silencio -- o bug do PHE


def consolida_com_checagem(blocos, limite=LIMITE_DE_LINHAS):
    pytest.fail("TODO: consolide e levante ValueError se alguma linha for perdida")


BLOCOS = [[1, 2, 3, 4, 5, 6], [7, 8, 9, 10, 11, 12]]  # 12 linhas, limite 10


# TODO 1: comprove o bug -- entram 12 linhas, saem 10, sem erro nenhum.
def test_consolidacao_perde_linhas_em_silencio():
    pytest.fail("TODO: assert que a saida tem menos linhas que a entrada e nada foi levantado")


# TODO 2: a versao com checagem precisa recusar o truncamento.
def test_consolidacao_com_checagem_rejeita_perda():
    pytest.fail("TODO: use pytest.raises(ValueError, match='perdidas') com BLOCOS")


# TODO 3: quando tudo cabe, a checagem deixa passar e preserva as linhas.
def test_consolidacao_com_checagem_aceita_quando_cabe():
    pytest.fail("TODO: use blocos que somem <= LIMITE_DE_LINHAS e verifique a saida completa")

---
## 9. [Prática 5] Validação cruzada - o sensor único do 737 MAX

O MCAS agia com base em **um** sensor de ângulo de ataque. O avião tinha dois; nada
comparava um com o outro. Quando um passou a mentir, não havia como saber.

**Sua tarefa:** implementar `check_sensores_concordam` e a leitura segura que só aceita
o valor quando os dois sensores batem.

In [ ]:
%%ipytest

TOLERANCIA_GRAUS = 5.0


def check_sensores_concordam(a, b, tolerancia=TOLERANCIA_GRAUS):
    pytest.fail("TODO: levante ValueError('sensores divergem...') se |a - b| > tolerancia")


def leitura_segura(a, b, tolerancia=TOLERANCIA_GRAUS):
    pytest.fail("TODO: valide e devolva a media dos dois sensores")


# TODO 1: dois sensores saudaveis (12.0 e 12.4) tem que passar.
def test_sensores_proximos_sao_aceitos():
    pytest.fail("TODO: assert check_sensores_concordam(12.0, 12.4) is True")


# TODO 2: o cenario do acidente -- um sensor travado em valor absurdo.
def test_sensor_travado_e_rejeitado():
    pytest.fail("TODO: use pytest.raises com 12.0 e 75.0")


# TODO 3: a leitura segura devolve a media quando os dois concordam.
def test_leitura_segura_devolve_media():
    pytest.fail("TODO: leitura_segura(10.0, 11.0) == 10.5")


# TODO 4: a leitura segura NAO devolve numero nenhum quando eles divergem.
def test_leitura_segura_nao_inventa_valor():
    pytest.fail("TODO: pytest.raises com 10.0 e 80.0")

---
## 10. [Prática 6] Procedência do dado - o caso IBM Watson

O Watson for Oncology recomendava tratamento com base em **casos hipotéticos**, não em
histórico real de pacientes. Nenhuma métrica de modelo pega isso: o dado estava limpo,
bem formatado e completo. Só não era o que todo mundo assumia que era.

**Sua tarefa:** escrever a checagem que recusa treinar com dado que não é real.

In [ ]:
%%ipytest

ORIGENS_ACEITAS = {"real"}


def check_procedencia(df, origens_aceitas=ORIGENS_ACEITAS):
    pytest.fail("TODO: exija a coluna 'origem' e recuse qualquer valor fora de origens_aceitas")


df_real = pd.DataFrame({"paciente": [1, 2], "origem": ["real", "real"]})
df_misto = pd.DataFrame({"paciente": [1, 2], "origem": ["real", "hipotetico"]})
df_sem_coluna = pd.DataFrame({"paciente": [1, 2]})


# TODO 1: dado 100% real passa.
def test_procedencia_aceita_dado_real():
    pytest.fail("TODO: assert check_procedencia(df_real) is True")


# TODO 2: um unico registro hipotetico ja reprova o lote inteiro.
def test_procedencia_rejeita_caso_hipotetico():
    pytest.fail("TODO: pytest.raises com df_misto")


# TODO 3: ausencia da coluna 'origem' e falha, nao "passa por omissao".
def test_procedencia_exige_a_coluna():
    pytest.fail("TODO: pytest.raises com df_sem_coluna")

---
## Fechamento

Cada prática de hoje é a checagem que faltou num caso real da abertura:

| Prática | Caso | O que o teste garante |
|---|---|---|
| 3 | Mars Climate Orbiter (1999) | as duas pontas falam a mesma unidade |
| 4 | Public Health England (2020) | nenhuma linha some no caminho |
| 5 | Boeing 737 MAX (2018–19) | a entrada é confirmada por uma segunda fonte |
| 6 | IBM Watson for Oncology (2018) | o dado é o que você assume que é |

Nenhuma dessas checagens tem mais que cinco linhas. É esse o ponto.